In [1]:
from glob import glob 

for g in glob('./data/*.pdf'):
    print(g)

./data\2040_seoul_plan.pdf
./data\OneNYC_2050_Strategic_Plan.pdf
./data\프롬프트 명령어_한글.pdf


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
        pdf_path (str): PDF 파일의 경로.
        chunk_size (int, 선택적): 각 텍스트 청크의 크기. 기본값은 1000입니다.
        chunk_overlap (int, 선택적): 청크 간의 중첩 크기. 기본값은 100입니다.
    반환값:
        list: 분할된 텍스트 청크의 리스트.
    """
    print(f"PDF: {pdf_path} -----------------------------")

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)
    
    print(f"Number of splits: {len(splits)}\n")
    return splits


In [ ]:
%pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/6.1 MB ? eta -:--:--
   ---------------------------------------- 6.1/6.1 MB 42.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 46.0 MB/s eta 0:00:55
   ---------------------------------------- 0.0/2.5 GB 47.7 MB/s eta 0:00:53
   ---------------------------------------- 0.0/2.5 GB 49.5 MB/s eta 0:00:51
    --------------------------------------- 0.0/2.5 GB 51.5 MB/s eta 0:00:49
    --------------------------------------- 0.1/2.5 GB 51.8 MB/s eta 0:00:48
   - -------------------------------------- 0.1/2.5 GB 51.6 MB/s eta 0:00:48
   - -------------------------------------- 0.1/2.5 GB 52.3 MB/s eta 0:00:47
   - -----------------------------------

  You can safely remove it manually.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs = {'device': 'cpu'}, # cuda 지원하면 cuda, 아닌 경우 cpu
    encode_kwargs = {'normalize_embeddings': True}, # 임베딩 정규화
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
embeddings.embed_documents("안녕하세요")

[[0.002456831047311425,
  0.0322626568377018,
  -0.007424292620271444,
  0.00526847131550312,
  -0.05809730291366577,
  -0.030428674072027206,
  -0.008300005458295345,
  0.036742474883794785,
  0.009622779674828053,
  -0.007633916102349758,
  0.017204467207193375,
  0.038877058774232864,
  -0.02964903600513935,
  -0.021820029243826866,
  -0.0008699100580997765,
  -0.032562505453825,
  0.03178011626005173,
  -0.022462930530309677,
  0.023502597585320473,
  -0.012333741411566734,
  -0.038810186088085175,
  -0.017907707020640373,
  0.038729410618543625,
  0.013715686276555061,
  0.025562595576047897,
  0.0229959636926651,
  -0.02759172022342682,
  0.027073416858911514,
  -0.009775801561772823,
  -0.026509825140237808,
  -0.006837860215455294,
  -0.02930246852338314,
  0.028823185712099075,
  -0.07535599172115326,
  -0.03274378553032875,
  -0.004341255873441696,
  -0.023116106167435646,
  0.02154514007270336,
  -0.054298121482133865,
  0.05281256139278412,
  0.04333128780126572,
  -0.02088

In [5]:
from langchain_chroma import Chroma

import os
persist_directory='./llama/chroma_store'

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, 
        embedding_function=embeddings
    )
else:
    print("Creating new Chroma store")
    
    all_splits = []
    for g in glob('./data/*.pdf'):
        all_splits.extend(read_pdf_and_split_text(g))

    print(f"Total number of splits: {len(all_splits)}")

    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embeddings,
        persist_directory=persist_directory
    )

Creating new Chroma store
PDF: ./data\2040_seoul_plan.pdf -----------------------------
Number of splits: 308

PDF: ./data\OneNYC_2050_Strategic_Plan.pdf -----------------------------
Number of splits: 1023

PDF: ./data\프롬프트 명령어_한글.pdf -----------------------------
Number of splits: 33

Total number of splits: 1364


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chunks = retriever.invoke("서울시 쓰레기 저감 정책")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)

{'moddate': '2024-12-12T18:16:11+09:00', 'page': 64, 'creationdate': '2024-12-12T18:16:11+09:00', 'source': './data\\2040_seoul_plan.pdf', 'total_pages': 205, 'pdfversion': '1.4', 'author': 'SI', 'page_label': '65', 'creator': 'Hwp 2020 11.0.0.5178', 'producer': 'Hancom PDF 1.3.0.542'}
제3절 2040 서울도시기본계획 7대 목표57Ÿ특히, 서울시는 현재 온실가스 배출량의 90%를 차지하고 있는 건물과 수송 부문 감축을 위해 적극적인 대책을 마련하고 있다.-2026년까지 건물 에너지효율화사업 100만 호를 추진, 건물온실가스총량제, 신규건물 제로에너지 건물(ZEB) 의무화를 통해 기존 및 신규건물의 제로에너지화를 촉진-수송 부문 배출감축을 위해 전기차 비중을 2026년까지 10%(’21년 5.2만 대 → ’26년 40만 대)로 확대하고, 22만기의 충전 인프라 구축을 계획 중(’21년 2만기) 시민 생활과 안전을 위한 기후위기 대응 요구 심화Ÿ서울은 인구, 시설 등이 밀집해 있는 대도시로 기온 상승, 폭염, 집중호우, 태풍, 한파 등 극한 기후 현상이 더욱 빈번하게 발생할 것으로 전망된다. 이러한 기후위험은 서울시민의 일상생활과 안전을 크게 위협할 수 있다. 따라서 시민의 일상을 보호하고 도시의 회복력을 강화하기 위한 적극적인 기후위기 대응 전략이 필요하다.Ÿ특히 다양화·복합화되는 재난안전사고에 대응해 전통적인 자연·사회재난의 범주뿐 아니라, 신종 복합재난까지 대비할 수 있는 다면적인 대응체계 마련이 요구된다.2) 추진전략탄소중립·기후위기 적응대책은 시의 모든 정책과 사업의 주요 원칙으로 고려Ÿ탄소중립은 돌이키기 어려운 기후재난을 막기 위한 국제사회와의 약속이기에 시의 모든 정책과 사업의 결정 과정에서 주요하게 고려해야 할 포괄적 ‘원칙’으